In [ ]:

# Financial metrics loader.(2nd code processing)
# reads ZIP only,upload processed Renamed files only

import pandas as pd
import numpy as np
import re
import zipfile
from google.colab import files
import io
import os
!pip install xlsxwriter

def detect_currency(file_path):
    """
    Scan entire workbook for presence of 'INR' or 'KINR' (case-insensitive).
    Returns 'INR', 'KINR', or 'Unknown'.
    """
    xl = pd.ExcelFile(file_path)
    for sheet_name in xl.sheet_names:
        df = xl.parse(sheet_name, header=None)
        for row in df.itertuples(index=False):
            for cell in row:
                if pd.isna(cell):
                    continue
                text = str(cell).strip().upper()
                if 'KINR' in text or 'KNIR' in text:
                    print(f"Detected currency KINR in sheet '{sheet_name}' cell '{text}'")
                    return 'KINR'
                if 'INR' in text and 'KINR' not in text:
                    print(f"Detected currency INR in sheet '{sheet_name}' cell '{text}'")
                    return 'INR'
    print("Currency not explicitly detected, returning 'Unknown'")
    return 'Unknown'

def find_intersection_value(df, row_keyword, col_keyword):
    row_idx = None
    col_idx = None

    # Escape keywords for literal matching
    escaped_row_keyword = re.escape(row_keyword.strip().lower())
    escaped_col_keyword = re.escape(col_keyword.strip().lower())

    # Find row index
    for idx in range(len(df)):
        row_values = df.iloc[idx].astype(str).fillna('').str.strip().str.lower()
        if row_values.str.contains(escaped_row_keyword, regex=True).any():
            row_idx = idx
            print(f"Found row '{row_keyword}' at index {row_idx}")
            break
    if row_idx is None:
        print(f"Warning: row '{row_keyword}' not found")
        return None

    # Find column index
    if col_keyword.lower() == "total":
        for r in range(min(30, len(df))):
            row_values = df.iloc[r].astype(str).fillna('').str.strip().str.lower()
            for c, val in enumerate(row_values):
                if val == "total":
                    col_idx = c
                    print(f"Found column 'Total' at row {r}, col {c}")
                    break
            if col_idx is not None:
                break
    else:
        for r in range(len(df)):
            row_values = df.iloc[r].astype(str).fillna('').str.strip().str.lower()
            if row_values.str.contains(escaped_col_keyword, regex=True).any():
                col_idx = row_values[row_values.str.contains(escaped_col_keyword, regex=True)].index[0]
                print(f"Found column '{col_keyword}' at row {r}, col {col_idx}")
                break
    if col_idx is None:
        print(f"Warning: column '{col_keyword}' not found")
        return None

    val = df.iat[row_idx, col_idx]
    if pd.isna(val) or isinstance(val, str):
        for offset in range(-3, 4):
            test_col = col_idx + offset
            if 0 <= test_col < len(df.columns):
                test_val = df.iat[row_idx, test_col]
                if pd.notna(test_val) and isinstance(test_val, (int, float, np.integer, np.floating)):
                    val = test_val
                    break
    print(f"Value found for '{row_keyword}' vs '{col_keyword}': {val}")
    return val

def find_intersection_value_subcol(df, row_keyword, col_keyword, sub_col_keyword):
    row_idx = None
    escaped_row_keyword = re.escape(row_keyword.strip().lower())
    escaped_col_keyword = re.escape(col_keyword.strip().lower())
    escaped_sub_col_keyword = re.escape(sub_col_keyword.strip().lower())

    # Find row index
    for idx in range(len(df)):
        row_values = df.iloc[idx].astype(str).fillna('').str.strip().str.lower()
        if row_values.str.contains(escaped_row_keyword, regex=True).any():
            row_idx = idx
            print(f"Found row '{row_keyword}' at index {row_idx}")
            break
    if row_idx is None:
        print(f"Warning: row '{row_keyword}' not found")
        return None

    # Find main column header
    main_col_idx = None
    main_header_row = None
    for r in range(min(30, len(df))):
        row_values = df.iloc[r].astype(str).fillna('').str.strip().str.lower()
        if row_values.str.contains(escaped_col_keyword, regex=True).any():
            main_col_idx = row_values[row_values.str.contains(escaped_col_keyword, regex=True)].index[0]
            main_header_row = r
            print(f"Found main column '{col_keyword}' at row {r}, col {main_col_idx}")
            break
    if main_col_idx is None:
        print(f"Warning: main column '{col_keyword}' not found")
        return None

    sub_header_start = main_header_row + 1
    sub_header_end = min(main_header_row + 5, len(df))

    columns_under_main = []
    for c in range(main_col_idx, len(df.columns)):
        has_subheader = False
        for r in range(sub_header_start, sub_header_end):
            cell_val = df.iat[r, c]
            if pd.notna(cell_val) and str(cell_val).strip():
                has_subheader = True
                break
        if has_subheader:
            columns_under_main.append(c)
        else:
            break

    if not columns_under_main:
        print(f"Warning: no sub-columns under '{col_keyword}'")
        return None

    col_idx = None
    for r in range(sub_header_start, sub_header_end):
        for c in columns_under_main:
            cell_val = df.iat[r, c]
            if pd.notna(cell_val) and re.search(escaped_sub_col_keyword, str(cell_val).strip().lower()):
                col_idx = c
                print(f"Found sub-column '{sub_col_keyword}' at col {c}, row {r}")
                break
        if col_idx is not None:
            break

    if col_idx is None:
        print(f"Warning: sub-column '{sub_col_keyword}' not found under '{col_keyword}'")
        return None

    val = df.iat[row_idx, col_idx]
    if pd.isna(val) or isinstance(val, str):
        for offset in range(-3, 4):
            test_col = col_idx + offset
            if 0 <= test_col < len(df.columns):
                test_val = df.iat[row_idx, test_col]
                if pd.notna(test_val) and isinstance(test_val, (int, float, np.integer, np.floating)):
                    val = test_val
                    break
    print(f"Value found for '{row_keyword}' vs '{col_keyword}' sub-column '{sub_col_keyword}': {val}")
    return val

def extract_financial_metrics(file_path, file_name):
    xl = pd.ExcelFile(file_path)
    eas_sheets = [s for s in xl.sheet_names if 'EAS' in s.upper()]
    sheet_name = eas_sheets[0] if eas_sheets else xl.sheet_names[0]
    print(f"Loading sheet: {sheet_name}")
    df = xl.parse(sheet_name, header=None)

    metrics = {}
    metrics['Currency'] = detect_currency(file_path)

    # ── Selling Price variations ──────────────────────────────────────────────
    # 'Selling Price' is the most common label; prefixed variants tried after.
    # Kept intentionally tight to avoid clashing with unrelated row labels.
    sp_variations = [
        'Selling Price',
        'Total Selling Price',
        'Net Selling Price',
    ]

    # ── DVC Material variations ───────────────────────────────────────────────
    dvm_variations = [
        'Total DVC Material (121 to 129)',
        'Total DVC Materials (121 to 129)',
        'DVC Material (121 to 129)',
        'DVC Materials (121 to 129)',
        'Total DVC Material',
        'Total DVC Materials'
    ]

    # ── DVC Labour variations ─────────────────────────────────────────────────
    dvl_variations = [
        'Total DVC Labour (131 to 149)',
        'Total DVC Labor (131 to 149)',
        'DVC Labour (131 to 149)',
        'DVC Labor (131 to 149)',
        'Total DVC Labour',
        'Total DVC Labor'
    ]

    # ── DVC Material metrics ──────────────────────────────────────────────────
    metrics['DVC M % on Sales'] = None
    for v in dvm_variations:
        val = find_intersection_value(df, v, 'vs. Sales')
        if val is not None:
            metrics['DVC M % on Sales'] = val
            print(f"Found DVC Material % on Sales: {val}")
            break

    metrics['DVC M'] = None
    for v in dvm_variations:
        val = find_intersection_value(df, v, 'Total')
        if val is not None:
            metrics['DVC M'] = val
            print(f"Found DVC Material Total: {val}")
            break

    # ── DVC Labour metrics ────────────────────────────────────────────────────
    metrics['DVC L % on Sales'] = None
    for v in dvl_variations:
        val = find_intersection_value(df, v, 'vs. Sales')
        if val is not None:
            metrics['DVC L % on Sales'] = val
            break

    metrics['DVC L'] = None
    for v in dvl_variations:
        val = find_intersection_value(df, v, 'Total')
        if val is not None:
            metrics['DVC L'] = val
            break

    # ── Remaining existing metrics ────────────────────────────────────────────
    metrics['DVC O %'] = find_intersection_value(df, 'Other DVC (151 to 179)', 'vs. Sales')
    metrics['DVC O'] = find_intersection_value(df, 'Other DVC (151 to 179)', 'Total')

    metrics['MBC %'] = find_intersection_value(df, 'Total MBC Costs (201 to 299)', 'vs. Sales')
    metrics['MBC'] = find_intersection_value(df, 'Total MBC Costs (201 to 299)', 'Total')

    metrics['COGS Tendered'] = find_intersection_value(df, 'Total Cost of Goods Sold (COGS) (100+200)', 'Total')
    metrics['GM'] = find_intersection_value(df, 'Gross Margin (900-300)', 'Total')
    metrics['GM %'] = find_intersection_value(df, 'Gross Margin (900-300)', 'vs. Sales')

    # ── Selling Price (after GM %) ────────────────────────────────────────────
    metrics['Selling Price'] = None
    for v in sp_variations:
        val = find_intersection_value(df, v, 'Total')
        if val is not None:
            metrics['Selling Price'] = val
            print(f"Found Selling Price: {val}")
            break

    # ── Sub-column metrics ────────────────────────────────────────────────────
    metrics['Upstream Gross Margin %'] = find_intersection_value_subcol(
        df, 'Upstream Gross Margin Received from BO', 'Upstream P&L', '%'
    )

    metrics['Incoming Freight %'] = find_intersection_value(
        df, 'Incoming Freight', 'Coefficient and basis'
    )

    # Strip folder path, file extension, then any trailing decimal (e.g. 227656298.0 -> 227656298)
    so_raw = os.path.splitext(os.path.basename(file_name))[0]
    so_clean = re.sub(r'\.0+$', '', so_raw).strip()
    metrics['Sales Order No'] = so_clean
    return metrics

def adjust_value(row, col):
    """Row-wise currency adjustment helper function"""
    val = row[col]
    currency = row['Currency'].strip().upper() if pd.notna(row.get('Currency')) else 'KINR'

    if pd.isna(val) or not isinstance(val, (int, float, np.integer, np.floating)):
        return val

    if currency == 'KINR':
        return abs(val) * 1000
    else:  # INR or others - no multiplication
        return abs(val)

def create_output_excel(metrics_df, output_filename='extracted_metrics.xlsx'):
    percent_cols = [
        'DVC M % on Sales',        # DVC Material first
        'DVC L % on Sales',
        'DVC O %',
        'MBC %',
        'GM %',
        'Upstream Gross Margin %',
        'Incoming Freight %'
    ]
    kinr_cols = [
        'DVC M',                   # DVC Material first
        'DVC L',
        'DVC O',
        'MBC',
        'COGS Tendered',
        'GM',
        'Selling Price',           # Selling Price treated as absolute value (INR/KINR adjusted)
    ]

    # ── Enforce desired column order in output sheet ──────────────────────────
    ordered_cols = [
        'Sales Order No',
        'Currency',
        'DVC M % on Sales',
        'DVC M',
        'DVC L % on Sales',
        'DVC L',
        'DVC O %',
        'DVC O',
        'MBC %',
        'MBC',
        'COGS Tendered',
        'GM',
        'GM %',
        'Selling Price',
        'Upstream Gross Margin %',
        'Incoming Freight %',
    ]

    df = metrics_df.copy()

    # Keep only columns that exist, preserve order, append any unexpected extras at end
    existing_ordered = [c for c in ordered_cols if c in df.columns]
    extra_cols = [c for c in df.columns if c not in ordered_cols]
    df = df[existing_ordered + extra_cols]

    # Debug print currency distribution
    print("Currencies in combined data:")
    print(df['Currency'].value_counts())

    # Process Percent columns (normalize to 0-1 range)
    for col in percent_cols:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: abs(x) if pd.notna(x) else x)
            df[col] = df[col].apply(lambda x: x / 100 if (pd.notna(x) and x > 1) else x)

    # Apply row-wise currency adjustment for absolute value columns
    for col in kinr_cols:
        if col in df.columns:
            print(f"Applying per-row currency adjustment for '{col}'")
            df[col] = df.apply(lambda row: adjust_value(row, col), axis=1)

    try:
        import xlsxwriter
    except ImportError:
        print("Installing xlsxwriter...")
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "xlsxwriter"])
        import xlsxwriter

    # Create Excel output with formatting
    with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
        df.to_excel(writer, index=False, sheet_name='Metrics')

        workbook = writer.book
        worksheet = writer.sheets['Metrics']

        # Header formatting
        header_format = workbook.add_format({
            'bold': True,
            'bg_color': '#8ED973',
            'font_color': '#006100',
            'border': 1
        })

        for col_num, value in enumerate(df.columns):
            worksheet.write(0, col_num, value, header_format)

        # Column formatting
        percent_format = workbook.add_format({'num_format': '0.00%'})
        kinr_format = workbook.add_format({'num_format': '#,##0'})

        for i, col in enumerate(df.columns):
            max_len = max(df[col].astype(str).map(len).max(), len(col))
            if col in percent_cols:
                worksheet.set_column(i, i, max_len + 3, percent_format)
            elif col in kinr_cols:
                worksheet.set_column(i, i, max_len + 3, kinr_format)
            else:
                worksheet.set_column(i, i, max_len + 3)

    print(f"\nOutput saved to: {output_filename}")
    print("\nExtracted Metrics:")
    print("=" * 50)
    print(df.head())

    return df

def process_zip_file(zip_path):
    """Extract and process all Excel files from a zip file"""
    excel_files = []

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        file_list = zip_ref.namelist()
        print(f"Found {len(file_list)} files in zip")

        for file_name in file_list:
            if file_name.lower().endswith(('.xlsx', '.xls')):
                print(f"Processing Excel file from zip: {file_name}")
                try:
                    file_data = zip_ref.read(file_name)
                    excel_file = io.BytesIO(file_data)

                    metrics = extract_financial_metrics(excel_file, file_name)

                    currency = metrics.get('Currency', 'Unknown')
                    print(f"Detected currency for '{file_name}': '{currency}'")

                    preview = {k: metrics[k] for k in list(metrics)[:5]}
                    print(f"Metric preview for '{file_name}': {preview}")

                    excel_files.append(metrics)

                except Exception as e:
                    print(f"Error processing {file_name}: {str(e)}")

    return excel_files

def main():
    print("Friendly neighbourhood intern needs ZIP files:")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded.")
        return

    all_metrics = []

    for zip_filename in uploaded.keys():
        print(f"\nProcessing ZIP file: {zip_filename}")
        try:
            zip_metrics = process_zip_file(zip_filename)
            all_metrics.extend(zip_metrics)
            print(f"Extracted metrics from {len(zip_metrics)} Excel files in {zip_filename}")

        except Exception as e:
            print(f"Error processing ZIP file {zip_filename}: {str(e)}")

    if not all_metrics:
        print("No Excel files found or processed successfully.")
        return

    combined_df = pd.DataFrame(all_metrics)

    print("\nCurrencies found in combined DataFrame:")
    print(combined_df['Currency'].value_counts())
    print("Unique currencies:", combined_df['Currency'].unique())
    print(f"Combined DataFrame shape: {combined_df.shape}")
    print(f"Combined DataFrame columns: {combined_df.columns.tolist()}")

    output_filename = 'combined_extracted_metrics.xlsx'
    result_df = create_output_excel(combined_df, output_filename)

    files.download(output_filename)
    print("Fam, process completed")
    return result_df

if __name__ == "__main__":
    result = main()
